In [2]:
import pandas as pd

df1 = pd.read_csv("books.csv")
df2 = pd.read_csv("books_1.Best_Books_Ever.csv")


print("Length of dataset1:", len(df1))
print("Length of dataset2:", len(df2))


chunk_size = 100_000
total_rows = 0

for chunk in pd.read_csv("Books_rating.csv", chunksize=chunk_size):
    total_rows += len(chunk)

print("✅ Total rows in Books_rating.csv:", total_rows)

Length of dataset1: 1247
Length of dataset2: 52478
✅ Total rows in Books_rating.csv: 3000000


In [3]:
cols = pd.read_csv("Books_rating.csv", nrows=0).columns.tolist()
print("📚 Columns in Books_rating.csv:")
print(cols)


📚 Columns in Books_rating.csv:
['Id', 'Title', 'Price', 'User_id', 'profileName', 'review/helpfulness', 'review/score', 'review/time', 'review/summary', 'review/text']


In [ ]:

for df in [df1, df2]:
    df["title"] = df["title"].str.lower().str.strip()
    df["author"] = df["author"].str.lower().str.strip()

merged_books = pd.merge(df1, df2, on=["author", "title"], how="outer")

chunk_size = 100_000
results = []

for chunk in pd.read_csv("Books_rating.csv", chunksize=chunk_size):
    
    chunk["Title"] = chunk["Title"].str.lower().str.strip()
    chunk.rename(columns={"Title": "title"}, inplace=True)

    
    merged_chunk = pd.merge(merged_books, chunk, on="title", how="inner")
    results.append(merged_chunk)


final_df = pd.concat(results, ignore_index=True)

print(" Final merged shape:", final_df.shape)



✅ Final merged shape: (969028, 40)


In [8]:
final_df.to_csv("final_merged_books.csv", index=False)

In [12]:
print(final_df.sample(5))

        Unnamed: 0                                              title  \
442389         NaN                              sorcerer's apprentice   
823244         NaN                                       earth abides   
373615         NaN                                     to be the best   
838220         NaN  matter and consciousness: a contemporary intro...   
320017         NaN                                         persuasion   

                                                   author  release year  \
442389                      tahir shah (goodreads author)           NaN   
823244                                  george r. stewart           NaN   
373615         barbara taylor bradford (goodreads author)           NaN   
838220                                 paul m. churchland           NaN   
320017  jane austen, deidre shauna lynch (introduction...           NaN   

       synopsis  book length  rating_x  number of ratings  \
442389      NaN          NaN       NaN           

In [13]:
from tabulate import tabulate

print(tabulate(final_df.sample(5), headers='keys', tablefmt='pretty'))

+--------+------------+---------------------------------------+--------------------------------------------------------------------------------------------------------------------------+--------------+----------+-------------+----------+-------------------+-----------------------------------------+-----------------+----------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
import pandas as pd


final_df = pd.read_csv('final_merged_books.csv', low_memory=False)


print(f" Final dataset shape: {final_df.shape}")


final_df['title'] = final_df['title'].astype(str).str.lower().str.strip()
final_df['author'] = final_df['author'].fillna('').astype(str).str.lower().str.strip()



✅ Final dataset shape: (969028, 40)


In [ ]:

unique_title_author = final_df[['title', 'author']].drop_duplicates()

print(f"Unique (title, author) combinations: {unique_title_author.shape[0]}")


✅ Unique (title, author) combinations: 7101


In [ ]:
num_unique_authors = final_df['author'].nunique()
print(f" Number of unique authors: {num_unique_authors}")


🖋️ Number of unique authors: 4848


In [ ]:

missing_values = final_df.isna().sum().sort_values(ascending=False)
print("❓ Missing values per column:\n", missing_values)


❓ Missing values per column:
 Unnamed: 0            969028
release year          969028
synopsis              969028
book length           969028
rating_x              969028
number of ratings     969028
Price                 896990
edition               746156
series                589452
profileName           216824
User_id               216797
price                 196865
firstPublishDate       98462
language               24526
publisher              20648
pages                  11576
description            10491
publishDate             3032
bookFormat              1919
likedPercent             528
coverImg                 523
review/summary           156
review/text                1
numRatings                 0
rating_y                   0
title                      0
review/time                0
review/score               0
review/helpfulness         0
author                     0
bookId                     0
Id                         0
ratingsByStars             0
isbn         

In [27]:
columns_to_drop = [
    'Unnamed: 0','profileName', 'User_id', 'pages', 'release year', 'synopsis', 'book length', 
    'rating_x', 'number of ratings', 'Price', 'price', 'edition'
]

final_df_clean = final_df.drop(columns=columns_to_drop)
print(f"New shape after dropping: {final_df_clean.shape}")


New shape after dropping: (969028, 28)


In [12]:


reviews_per_title_author = final_df.groupby(['title', 'author']).size().reset_index(name='review_count').sort_values(by='review_count', ascending=False)

# Display top 10
print("📚 Reviews per (title, author) combination (top 10):\n", reviews_per_title_author.head(10))


📚 Reviews per (title, author) combination (top 10):
                                       title  \
3707                    pride and prejudice   
464                          atlas shrugged   
7069                      wuthering heights   
6596                  to kill a mockingbird   
3349                        of mice and men   
2767                           little women   
2001                     great expectations   
5300                              the giver   
2054  harry potter and the sorcerer's stone   
4885                 the catcher in the rye   

                                                 author  review_count  
3707          jane austen, anna quindlen (introduction)         20371  
464   ayn rand, leonard peikoff (goodreads author) (...         12513  
7069  emily brontë, richard j. dunn (editor), david ...         10780  
6596                                         harper lee          9601  
3349                                     john steinbeck          9420

In [21]:

title_author_variation = final_df.groupby('title')['author'].nunique().reset_index(name='num_authors')


confusing_titles = title_author_variation[title_author_variation['num_authors'] > 5]

print(f"📚 Number of titles with multiple distinct authors listed: {confusing_titles.shape[0]}")


📚 Number of titles with multiple distinct authors listed: 39


In [15]:
# For each confusing title, list all its authors
confusing_titles_list = confusing_titles['title'].tolist()

messy_authors_per_title = final_df[final_df['title'].isin(confusing_titles_list)][['title', 'author']].drop_duplicates()

messy_authors_per_title.to_csv('messy_authors_review.csv', index=False)



In [43]:
import pandas as pd
import numpy as np

# Step 1: Load your dataset
final_df = pd.read_csv('final_merged_books_clean.csv', low_memory=False)

# Step 2: First Drop TRUE NaNs
final_df = final_df.dropna(how='any')



# Step 5: Final replace NaN with None for MySQL
final_df = final_df.where(pd.notnull(final_df), None)

# Step 6: Save clean
final_df.to_csv('final_merged_books_clean.csv', index=False)

print(f"✅ Final cleaned dataset shape: {final_df.shape}")

from tabulate import tabulate

print(tabulate(final_df.sample(5), headers='keys', tablefmt='pretty'))

✅ Final cleaned dataset shape: (311342, 28)
+--------+-------------------+---------------------------------+-------------------------+-----------------------------------+----------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:

df_clean = pd.read_csv('final_merged_books_clean.csv', low_memory=False)


df_clean['title'] = df_clean['title'].astype(str).str.lower().str.strip()
df_clean['author'] = df_clean['author'].astype(str).str.lower().str.strip()


num_unique_titles = df_clean['title'].nunique()
print(f"📚 Unique book titles: {num_unique_titles}")


num_unique_authors = df_clean['author'].nunique()
print(f"✍️ Unique authors: {num_unique_authors}")

num_unique_title_author = df_clean[['title', 'author']].drop_duplicates().shape[0]
print(f"📖 Unique (title, author) combinations: {num_unique_title_author}")


same_title_diff_authors = df_clean.groupby('title')['author'].nunique().reset_index()
same_title_diff_authors = same_title_diff_authors[same_title_diff_authors['author'] > 1]

print(f"🕵️ Titles shared by multiple authors: {same_title_diff_authors.shape[0]}")


missing_columns = df_clean.isna().sum()
print("\n🧹 Missing values per column (should all be 0):")
print(missing_columns[missing_columns > 0])


📚 Unique book titles: 1825
✍️ Unique authors: 1292
📖 Unique (title, author) combinations: 1969
🕵️ Titles shared by multiple authors: 110
📈 Average reviews per (title, author): 158.12

🏆 Top 10 most reviewed books (title, author):
                                        title  \
1852                    to kill a mockingbird   
780                              little women   
1497                                the giver   
596     harry potter and the sorcerer's stone   
1525     the hitchhiker's guide to the galaxy   
630                                     holes   
1736                             the stranger   
1737                             the stranger   
595   harry potter and the chamber of secrets   
597                                   hatchet   

                                                 author  review_count  
1852                                         harper lee          9598  
780                                   louisa may alcott          8322  
1497          

In [ ]:

same_title_isbn_diff_authors = (
    df_clean.groupby(['title', 'isbn'])['author']
    .nunique()
    .reset_index()
)


same_title_isbn_diff_authors = same_title_isbn_diff_authors[same_title_isbn_diff_authors['author'] > 1]


print(f"🕵️ Titles (with ISBN) shared by multiple authors: {same_title_isbn_diff_authors.shape[0]}")
print(same_title_isbn_diff_authors)


🕵️ Titles (with ISBN) shared by multiple authors: 0
Empty DataFrame
Columns: [title, isbn, author]
Index: []


In [49]:
import pymysql
import pandas as pd

conn = pymysql.connect(host='mysql.clarksonmsda.org', port=3306,
                       user='ia626', passwd='ia626clarkson',
                       db='ia626', autocommit=True)

cur = conn.cursor(pymysql.cursors.DictCursor)


In [50]:
drop_books_sql = "DROP TABLE IF EXISTS books;"
drop_reviews_sql = "DROP TABLE IF EXISTS reviews;"
drop_summaries_sql = "DROP TABLE IF EXISTS summaries;"

cur.execute(drop_reviews_sql)
cur.execute(drop_summaries_sql)
cur.execute(drop_books_sql)


create_books_sql = '''
CREATE TABLE IF NOT EXISTS books (
    book_id INT AUTO_INCREMENT PRIMARY KEY,
    title TEXT,
    author TEXT,
    isbn TEXT,
    publication_year TEXT,
    genre TEXT,
    cover_image TEXT
);
'''

create_summaries_sql = '''
CREATE TABLE IF NOT EXISTS summaries (
    summary_id INT AUTO_INCREMENT PRIMARY KEY,
    book_id INT,
    summary_text TEXT,
    source TEXT ,
    FOREIGN KEY (book_id) REFERENCES books(book_id)
);
'''

create_reviews_sql = '''
CREATE TABLE IF NOT EXISTS reviews (
    review_id INT AUTO_INCREMENT PRIMARY KEY,
    book_id INT,
    rating FLOAT,
    review_summary TEXT,
    review_text TEXT,
    source TEXT ,
    FOREIGN KEY (book_id) REFERENCES books(book_id)
);
'''

cur.execute(create_books_sql)
cur.execute(create_summaries_sql)
cur.execute(create_reviews_sql)


0

In [ ]:
import pandas as pd

chunk_size = 10000
book_chunk = []
book_map = {}


for chunk in pd.read_csv('final_merged_books_clean.csv', chunksize=chunk_size, low_memory=False):
    chunk['title'] = chunk['title'].astype(str).str.lower().str.strip()
    chunk['author'] = chunk['author'].fillna('').astype(str).str.lower().str.strip()

    for _, row in chunk.iterrows():
        title = row['title']
        author = row['author']
        key = (title, author)

        if key not in book_map:
            book_chunk.append((
                title,
                author,
                row.get('isbn'),
                row.get('publishDate'),
                str(row.get('genres')),
                row.get('coverImg')
            ))
            book_map[key] = None 

    if book_chunk:
        cur.executemany('''
            INSERT INTO books (title, author, isbn, publication_year, genre, cover_image)
            VALUES (%s, %s, %s, %s, %s, %s);
        ''', book_chunk)
        book_chunk = []


conn.commit()

cur.execute('SELECT book_id, title, author FROM books')
for row in cur.fetchall():
    book_map[(row['title'], row['author'])] = row['book_id']


In [ ]:
summary_chunk = []
review_chunk = []

for chunk in pd.read_csv('final_merged_books_clean.csv', chunksize=chunk_size, low_memory=False):
    chunk['title'] = chunk['title'].astype(str).str.lower().str.strip()
    chunk['author'] = chunk['author'].fillna('').astype(str).str.lower().str.strip()

    for _, row in chunk.iterrows():
        title = row['title']
        author = row['author']
        key = (title, author)
        book_id = book_map.get(key)

        if book_id:
           
            if pd.notna(row.get('description')):
                summary_chunk.append((
                    book_id,
                    row['description'],
                    'human'
                ))

            # Insert reviews if review_text exists
            if pd.notna(row.get('review/text')):
                review_chunk.append((
                    book_id,
    
                    row.get('review/score'),
                    row.get('review/summary'),
                    row.get('review/text'),
                    'human'
                ))

   
    if summary_chunk:
        cur.executemany('''
            INSERT INTO summaries (book_id, summary_text, source)
            VALUES (%s, %s, %s);
        ''', summary_chunk)
        summary_chunk = []

    # Insert reviews (✅ use %s for all fields!)
    if review_chunk:
        cur.executemany('''
            INSERT INTO reviews (book_id,  rating, review_summary, review_text, source)
            VALUES (%s, %s, %s, %s, %s);
        ''', review_chunk)
        review_chunk = []

# Final commit
conn.commit()

